# Market Dimensionality / Opportunity Set — Feasibility Test v0.3

## Objetivo desta versão

A v0.2 mostrou que, no universo bruto de 9 ETFs setoriais dos EUA, **Effective Rank (participation ratio), mean correlation e market mode ordenam os estados de mercado quase da mesma forma**.

A v0.3 faz duas coisas antes de tocar em alpha:

1. **audita matematicamente o Effective Rank**, deixando explícita sua redundância com correlações ao quadrado quando \(N\) é fixo;
2. testa se a estrutura continua essencialmente a mesma **depois de remover o fator de mercado (SPY)**.

> **Ainda não há backtest de estratégia nesta versão.**
> O objetivo é decidir se existe uma variável estrutural suficientemente defensável para justificar o próximo passo.

## Research question desta etapa

> Depois de retirar o movimento comum do mercado, sobra uma estrutura cross-sectional suficientemente distinta e estável para justificar a tese de *Opportunity Set*?

## Hipóteses

- **H1 — sanity estrutural:** o ER bruto mede concentração de co-movimento.
- **H2 — auditoria matemática:** o participation-ratio ER é uma transformação monotônica da correlação quadrática agregada para \(N\) fixo.
- **H3 — residualização:** retirar SPY deve reduzir o domínio do fator comum e pode revelar uma estrutura relativa diferente.
- **H4 — decisão:** só avançaremos para alpha se a estrutura residual acrescentar uma pergunta econômica útil.

## 0. Regras metodológicas congeladas

- Universo: `XLB, XLE, XLF, XLI, XLK, XLP, XLU, XLV, XLY`.
- Fator de mercado: `SPY`.
- Frequência: retornos diários; observação estrutural mensal.
- Janela-base: **252 pregões**.
- Nenhum dado posterior a \(t\) entra na estimativa em \(t\).
- OLS de mercado estimado dentro da própria janela histórica disponível em \(t\).
- Nenhum threshold de trading nesta versão.
- Nenhuma métrica será promovida porque “parece melhor” visualmente.

In [ ]:
from __future__ import annotations

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

try:
    import yfinance as yf
except ImportError as exc:
    raise ImportError("Instale yfinance antes de rodar: pip install yfinance") from exc

DATA_DIR = Path("../data/raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)

SECTOR_TICKERS = ["XLB", "XLE", "XLF", "XLI", "XLK", "XLP", "XLU", "XLV", "XLY"]
MARKET_TICKER = "SPY"
ALL_TICKERS = SECTOR_TICKERS + [MARKET_TICKER]

START = "2000-01-01"
END = None
WINDOW = 252
MIN_COVERAGE = 0.98

CACHE_FILE = DATA_DIR / "us_sector_etfs_plus_spy_adjusted_close.csv"

## 1. Dados

A v0.3 usa um novo cache porque agora precisamos também do SPY.

In [ ]:
def load_prices(force_download: bool = False) -> pd.DataFrame:
    if CACHE_FILE.exists() and not force_download:
        return pd.read_csv(CACHE_FILE, index_col=0, parse_dates=True).sort_index()

    raw = yf.download(
        ALL_TICKERS,
        start=START,
        end=END,
        auto_adjust=True,
        progress=False,
        group_by="column",
        threads=True,
    )

    if raw.empty:
        raise RuntimeError("Download retornou vazio.")

    if isinstance(raw.columns, pd.MultiIndex):
        if "Close" in raw.columns.get_level_values(0):
            prices = raw["Close"].copy()
        elif "Close" in raw.columns.get_level_values(1):
            prices = raw.xs("Close", axis=1, level=1).copy()
        else:
            raise KeyError("Coluna Close não encontrada.")
    else:
        prices = raw[["Close"]].rename(columns={"Close": ALL_TICKERS[0]})

    prices = prices.reindex(columns=ALL_TICKERS).sort_index()
    prices.to_csv(CACHE_FILE)
    return prices

prices = load_prices()

coverage = prices.notna().mean().sort_values()
display(coverage.to_frame("coverage"))

first_valid_dates = prices.apply(pd.Series.first_valid_index)
common_start = max(first_valid_dates)

px = prices.loc[common_start:].copy()
daily_coverage = px.notna().mean(axis=1)
px = px.loc[daily_coverage >= MIN_COVERAGE]
px = px.ffill(limit=3).dropna(how="any")

print("Common start:", common_start)
print("Observações:", len(px))
print("Início:", px.index.min())
print("Fim:", px.index.max())

assert set(ALL_TICKERS).issubset(px.columns)
assert not px.isna().any().any()

In [ ]:
log_returns = np.log(px / px.shift(1)).dropna()

sector_returns = log_returns[SECTOR_TICKERS].copy()
market_returns = log_returns[MARKET_TICKER].copy()

display(sector_returns.describe().T[["mean", "std", "min", "max"]])

# 2. Auditoria matemática do participation-ratio Effective Rank

Para uma matriz de correlação \(C\) com \(N\) ativos:

\[
ER =
\frac{(\sum_i \lambda_i)^2}
{\sum_i \lambda_i^2}
\]

Como \(\sum_i \lambda_i=N\) e

\[
\sum_i \lambda_i^2
=
\operatorname{tr}(C^2)
=
N + 2\sum_{i<j}\rho_{ij}^2,
\]

temos:

\[
ER =
\frac{N^2}
{N + 2\sum_{i<j}\rho_{ij}^2}.
\]

Se

\[
\overline{\rho^2}
=
\frac{2}{N(N-1)}
\sum_{i<j}\rho_{ij}^2,
\]

então:

\[
\boxed{
ER =
\frac{N}
{1+(N-1)\overline{\rho^2}}
}
\]

Portanto, para \(N\) fixo, esse ER é uma transformação monotônica da correlação quadrática agregada.

In [ ]:
def structural_metrics_from_corr(corr: np.ndarray) -> dict[str, float]:
    n = corr.shape[0]

    eigvals = np.linalg.eigvalsh(corr)
    eigvals = np.clip(eigvals, 0.0, None)
    eigvals = np.sort(eigvals)[::-1]

    total = eigvals.sum()
    er = (total ** 2) / np.square(eigvals).sum()
    market_mode = eigvals[0] / total

    upper = corr[np.triu_indices(n, k=1)]
    mean_corr = np.mean(upper)
    mean_abs_corr = np.mean(np.abs(upper))
    mean_sq_corr = np.mean(np.square(upper))
    rms_corr = np.sqrt(mean_sq_corr)

    er_from_mean_sq = n / (1.0 + (n - 1.0) * mean_sq_corr)

    return {
        "effective_rank": float(er),
        "market_mode": float(market_mode),
        "mean_corr": float(mean_corr),
        "mean_abs_corr": float(mean_abs_corr),
        "mean_sq_corr": float(mean_sq_corr),
        "rms_corr": float(rms_corr),
        "er_from_mean_sq": float(er_from_mean_sq),
    }

## 3. Estrutura bruta

O erro numérico entre `effective_rank` e `er_from_mean_sq` deve ser praticamente zero.

In [ ]:
def rolling_raw_structure(sector_rets: pd.DataFrame, window: int = 252) -> pd.DataFrame:
    rows = []

    for end in range(window, len(sector_rets) + 1):
        w = sector_rets.iloc[end-window:end]
        corr = w.corr().to_numpy()

        metrics = structural_metrics_from_corr(corr)
        metrics["date"] = sector_rets.index[end - 1]
        rows.append(metrics)

    return pd.DataFrame(rows).set_index("date")

raw_structure_daily = rolling_raw_structure(sector_returns, WINDOW)

identity_error = (
    raw_structure_daily["effective_rank"]
    - raw_structure_daily["er_from_mean_sq"]
).abs()

print("Máximo erro absoluto da identidade ER:", identity_error.max())
display(raw_structure_daily.tail())

# 4. Residualização point-in-time contra SPY

Para cada janela histórica terminando em \(t\), estimamos para cada setor:

\[
r_{i,\tau}
=
\alpha_{i,t}
+
\beta_{i,t}r_{SPY,\tau}
+
\epsilon_{i,\tau},
\qquad \tau \in [t-251,t].
\]

Depois usamos apenas os resíduos dessa janela histórica para construir a matriz de correlação residual.

Esta etapa **não assume** que residualização vai “salvar” o ER. Ela testa se o fator de mercado explica a redundância extrema observada nos retornos brutos.

In [ ]:
def residualize_window_against_market(
    sector_window: pd.DataFrame,
    market_window: pd.Series,
) -> pd.DataFrame:
    # OLS univariado com intercepto, estimado separadamente por ativo.
    m = market_window.to_numpy(dtype=float)
    m_mean = m.mean()
    m_centered = m - m_mean
    var_m = np.dot(m_centered, m_centered)

    if var_m <= 0:
        raise ValueError("Variância do fator de mercado é zero.")

    residuals = pd.DataFrame(
        index=sector_window.index,
        columns=sector_window.columns,
        dtype=float,
    )

    for ticker in sector_window.columns:
        y = sector_window[ticker].to_numpy(dtype=float)
        y_mean = y.mean()
        y_centered = y - y_mean

        beta = np.dot(m_centered, y_centered) / var_m
        alpha = y_mean - beta * m_mean

        fitted = alpha + beta * m
        residuals[ticker] = y - fitted

    return residuals


def rolling_residual_structure(
    sector_rets: pd.DataFrame,
    market_rets: pd.Series,
    window: int = 252,
) -> pd.DataFrame:
    aligned = sector_rets.join(market_rets.rename("market"), how="inner").dropna()
    sectors = aligned[sector_rets.columns]
    market = aligned["market"]

    rows = []

    for end in range(window, len(aligned) + 1):
        sector_w = sectors.iloc[end-window:end]
        market_w = market.iloc[end-window:end]

        residuals = residualize_window_against_market(sector_w, market_w)
        corr_resid = residuals.corr().to_numpy()

        metrics = structural_metrics_from_corr(corr_resid)
        metrics["date"] = aligned.index[end - 1]
        rows.append(metrics)

    return pd.DataFrame(rows).set_index("date")


resid_structure_daily = rolling_residual_structure(
    sector_returns,
    market_returns,
    WINDOW,
)

display(resid_structure_daily.tail())

## 5. Amostragem mensal preservando a data real

In [ ]:
def last_real_observation_each_month(df: pd.DataFrame) -> pd.DataFrame:
    month_key = df.index.to_period("M")
    return df.groupby(month_key, group_keys=False).tail(1).dropna()

raw_m = last_real_observation_each_month(raw_structure_daily)
resid_m = last_real_observation_each_month(resid_structure_daily)

print("RAW")
display(raw_m.tail())

print("RESIDUAL")
display(resid_m.tail())

# 6. Teste crítico: a residualização realmente muda a estrutura?

Comparamos Pearson e Spearman dentro das representações bruta e residual.

Se os descritores residuais continuarem quase perfeitamente redundantes, registramos isso; não inventamos uma narrativa.

In [ ]:
cols = [
    "effective_rank",
    "mean_corr",
    "mean_abs_corr",
    "mean_sq_corr",
    "rms_corr",
    "market_mode",
]

print("RAW — Pearson")
display(raw_m[cols].corr(method="pearson"))

print("RAW — Spearman")
display(raw_m[cols].corr(method="spearman"))

print("RESIDUAL — Pearson")
display(resid_m[cols].corr(method="pearson"))

print("RESIDUAL — Spearman")
display(resid_m[cols].corr(method="spearman"))

In [ ]:
comparison = pd.DataFrame({
    "raw_er": raw_m["effective_rank"],
    "resid_er": resid_m["effective_rank"],
    "raw_mean_corr": raw_m["mean_corr"],
    "resid_mean_corr": resid_m["mean_corr"],
    "raw_rms_corr": raw_m["rms_corr"],
    "resid_rms_corr": resid_m["rms_corr"],
    "raw_market_mode": raw_m["market_mode"],
    "resid_market_mode": resid_m["market_mode"],
}).dropna()

display(comparison.corr(method="pearson"))

## 7. Visualização bruto vs residual

Padronização apenas para visualização; não será usada em nenhum sinal.

In [ ]:
viz = (comparison - comparison.mean()) / comparison.std()

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(viz.index, viz["raw_er"], label="Raw ER")
ax.plot(viz.index, viz["resid_er"], label="Residual ER")
ax.set_title("Effective Rank bruto vs. residual")
ax.set_xlabel("Data")
ax.set_ylabel("Z-score (somente visualização)")
ax.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(viz.index, -viz["raw_rms_corr"], label="- Raw RMS Correlation")
ax.plot(viz.index, -viz["resid_rms_corr"], label="- Residual RMS Correlation")
ax.set_title("Dependência quadrática agregada: bruto vs. residual")
ax.set_xlabel("Data")
ax.set_ylabel("Z-score (somente visualização)")
ax.legend()
plt.show()

## 8. Diagnóstico resumido

In [ ]:
summary = pd.Series({
    "raw_ER_mean": raw_m["effective_rank"].mean(),
    "resid_ER_mean": resid_m["effective_rank"].mean(),

    "raw_mean_corr_mean": raw_m["mean_corr"].mean(),
    "resid_mean_corr_mean": resid_m["mean_corr"].mean(),

    "raw_rms_corr_mean": raw_m["rms_corr"].mean(),
    "resid_rms_corr_mean": resid_m["rms_corr"].mean(),

    "spearman_raw_ER_vs_meanCorr":
        raw_m["effective_rank"].corr(raw_m["mean_corr"], method="spearman"),

    "spearman_resid_ER_vs_meanCorr":
        resid_m["effective_rank"].corr(resid_m["mean_corr"], method="spearman"),

    "spearman_raw_ER_vs_RMS":
        raw_m["effective_rank"].corr(raw_m["rms_corr"], method="spearman"),

    "spearman_resid_ER_vs_RMS":
        resid_m["effective_rank"].corr(resid_m["rms_corr"], method="spearman"),

    "corr_rawER_vs_residER":
        comparison["raw_er"].corr(comparison["resid_er"]),

    "n_months": len(comparison),
})

display(summary)

# PONTO DE DECISÃO — não avance automaticamente

Depois de executar a v0.3, precisamos escolher conscientemente entre:

## Caminho A — Opportunity Set simples
Se a estrutura residual for economicamente diferente, mas o participation-ratio ER continuar redundante com medidas simples, abandonamos o branding “Effective Rank” e usamos a medida mais simples defensável.

## Caminho B — ampliar a representação espectral
Podemos formular uma hipótese nova com uma medida que use mais da distribuição dos autovalores, por exemplo:

\[
p_i = \frac{\lambda_i}{\sum_j \lambda_j},
\qquad
ER_{entropy} =
\exp\left(-\sum_i p_i\ln p_i\right)
\]

Isso **não é equivalente** ao participation ratio, mas seria uma nova decisão metodológica e não poderá ser escolhida a posteriori apenas por performance.

## Caminho C — abandonar Market Dimensionality
Se residualização não produzir distinção economicamente útil e tudo continuar essencialmente uma medida de correlação, encerramos a tese antes de construir o alpha.

## O que não faremos nesta versão
Não adicionaremos Residual Momentum, thresholds ou carteira antes dessa decisão.

## Registro de decisões — v0.3

| Item | Estado |
|---|---|
| Tese central | Market Dimensionality / Opportunity Set |
| Participation-ratio ER bruto | válido, porém fortemente redundante |
| Identidade ER ↔ mean squared correlation | auditada explicitamente |
| Fator comum | SPY |
| Residualização | rolling OLS point-in-time, 252 dias |
| Estrutura residual | **A TESTAR AO EXECUTAR** |
| Alpha | **NÃO IMPLEMENTADO** |
| OOS de alpha | **NÃO DEFINIDO AINDA** |
| Próxima decisão | Caminho A, B ou C |

# 9. v0.4 — Primeiro teste econômico: Opportunity Set → Residual Momentum

A partir daqui fazemos o primeiro teste que pode **salvar ou matar a tese**.

## Hipótese econômica pré-especificada

Depois de remover o fator de mercado, uma dependência residual menor significa que os setores estão se movendo de forma mais independente.

Definimos:

\[
D_t = RMSCorr^\epsilon_t
\]

e um *opportunity score* apenas para facilitar a leitura:

\[
O_t = -D_t.
\]

Assim:

- \(O_t\) alto = menor dependência residual;
- \(O_t\) baixo = maior dependência residual.

A hipótese primária é:

\[
O_t \uparrow
\Rightarrow
\text{maior eficácia futura do alpha cross-sectional}.
\]

Isso **não está provado**. É o que testaremos.

---

## Alpha pré-especificado: Residual Momentum 12–1

Para cada setor, em cada data de decisão \(t\):

1. estimamos \(\alpha_{i,t}\) e \(\beta_{i,t}\) contra SPY usando os últimos 252 pregões;
2. calculamos os resíduos históricos nessa janela;
3. excluímos os 21 pregões mais recentes;
4. somamos os resíduos restantes:

\[
RMOM_{i,t}
=
\sum_{\tau=t-251}^{t-21}
\epsilon_{i,\tau}.
\]

O lookback **12–1 foi congelado antes de observar qualquer resultado condicionado**.

### Controle
Também calculamos momentum bruto 12–1:

\[
MOM_{i,t}
=
\sum_{\tau=t-251}^{t-21}
r_{i,\tau}.
\]

O momentum bruto não é nossa tese; ele serve como benchmark.

## 10. Como medir “qualidade futura do alpha” sem escolher uma carteira arbitrária

Com apenas 9 ETFs, escolher “top 10%” ou “top 20%” seria artificial.

Por isso a métrica **primária** será o **cross-sectional Rank IC**:

\[
IC_t =
Spearman(
Signal_{i,t},
FutureReturn_{i,t+1}
).
\]

Interpretação:

- \(IC>0\): ativos com maior sinal tenderam a ter maior retorno relativo depois;
- \(IC=0\): ranking sem informação;
- \(IC<0\): ranking apontou na direção errada.

Isso testa **informação preditiva cross-sectional** antes de decidir pesos de carteira.

Como métrica econômica secundária, usamos:

\[
Spread_t =
\text{média dos 3 maiores sinais}
-
\text{média dos 3 menores sinais}.
\]

Com 9 ativos, isso equivale a tercis de 3 ativos e evita escolher um threshold após ver os resultados.

In [ ]:
# Especificação do alpha — congelada na v0.4

MOMENTUM_LOOKBACK = 252
MOMENTUM_SKIP = 21
TOP_K = 3

assert MOMENTUM_LOOKBACK == WINDOW, (
    "Nesta versão, a janela do momentum e a janela estrutural são ambas 252 dias."
)

## 11. Funções auxiliares para o alpha

O \(\beta_t\) usado para avaliar o mês seguinte fica **congelado na data da decisão**.

O retorno futuro beta-hedged de cada setor é:

\[
R^{hedged}_{i,t\to t+1}
=
\sum_{\tau=t+1}^{t_{next}}
\left(
r_{i,\tau}
-
\beta_{i,t} r_{SPY,\tau}
\right).
\]

Isso evita usar beta estimado com dados futuros para avaliar a decisão tomada em \(t\).

In [ ]:
def fit_market_model(
    sector_window: pd.DataFrame,
    market_window: pd.Series,
) -> tuple[pd.Series, pd.Series, pd.DataFrame]:
    """
    Estima alpha e beta por OLS, ativo a ativo, usando somente a janela histórica.
    Retorna alpha, beta e resíduos in-sample da janela.
    """
    m = market_window.to_numpy(dtype=float)
    m_mean = m.mean()
    m_centered = m - m_mean
    var_m = np.dot(m_centered, m_centered)

    if var_m <= 0:
        raise ValueError("Variância do fator de mercado é zero.")

    alphas = {}
    betas = {}
    residuals = pd.DataFrame(
        index=sector_window.index,
        columns=sector_window.columns,
        dtype=float,
    )

    for ticker in sector_window.columns:
        y = sector_window[ticker].to_numpy(dtype=float)
        y_mean = y.mean()
        y_centered = y - y_mean

        beta = np.dot(m_centered, y_centered) / var_m
        alpha = y_mean - beta * m_mean

        fitted = alpha + beta * m
        residuals[ticker] = y - fitted

        alphas[ticker] = alpha
        betas[ticker] = beta

    return pd.Series(alphas), pd.Series(betas), residuals


def spearman_cross_section(x: pd.Series, y: pd.Series) -> float:
    aligned = pd.concat([x.rename("x"), y.rename("y")], axis=1).dropna()
    if len(aligned) < 3:
        return np.nan
    return float(aligned["x"].corr(aligned["y"], method="spearman"))


def top_bottom_spread(
    signal: pd.Series,
    future_return: pd.Series,
    top_k: int = 3,
) -> float:
    aligned = pd.concat(
        [signal.rename("signal"), future_return.rename("future")],
        axis=1
    ).dropna()

    if len(aligned) < 2 * top_k:
        return np.nan

    ranked = aligned.sort_values("signal")
    bottom = ranked.head(top_k)["future"].mean()
    top = ranked.tail(top_k)["future"].mean()

    return float(top - bottom)

## 12. Construção do painel mensal de teste

Cada linha será uma decisão mensal.

Para a data \(t\), armazenamos:

- `resid_rms_corr`: intensidade da dependência residual observada em \(t\);
- `opportunity_score = -resid_rms_corr`;
- `resid_er`: transformação equivalente em participation-ratio ER;
- `raw_rms_corr`: controle estrutural bruto;
- `IC` do Residual Momentum no mês seguinte;
- `IC` do momentum bruto;
- spreads top 3 – bottom 3 para ambos os sinais.

A data da decisão é sempre uma data realmente observada.

In [ ]:
def build_monthly_alpha_panel(
    sector_rets: pd.DataFrame,
    market_rets: pd.Series,
    residual_structure_monthly: pd.DataFrame,
    raw_structure_monthly: pd.DataFrame,
    lookback: int = 252,
    skip: int = 21,
    top_k: int = 3,
) -> pd.DataFrame:
    aligned = sector_rets.join(
        market_rets.rename("market"),
        how="inner"
    ).dropna()

    sectors = aligned[sector_rets.columns]
    market = aligned["market"]

    decision_dates = residual_structure_monthly.index.intersection(
        raw_structure_monthly.index
    ).sort_values()

    # Precisamos de uma próxima data de decisão para medir o mês seguinte.
    rows = []

    for j in range(len(decision_dates) - 1):
        t = decision_dates[j]
        t_next = decision_dates[j + 1]

        if t not in aligned.index or t_next not in aligned.index:
            continue

        pos = aligned.index.get_loc(t)

        # Precisamos de lookback observações até e incluindo t.
        if not isinstance(pos, (int, np.integer)) or pos + 1 < lookback:
            continue

        hist = aligned.iloc[pos - lookback + 1 : pos + 1]
        sector_hist = hist[sector_rets.columns]
        market_hist = hist["market"]

        _, betas_t, residuals_hist = fit_market_model(
            sector_hist,
            market_hist,
        )

        # 12-1: exclui os últimos 21 pregões.
        residual_signal_window = residuals_hist.iloc[:-skip]
        raw_signal_window = sector_hist.iloc[:-skip]

        if residual_signal_window.empty or raw_signal_window.empty:
            continue

        resid_mom = residual_signal_window.sum(axis=0)
        raw_mom = raw_signal_window.sum(axis=0)

        # Retorno futuro: estritamente após t até a próxima data mensal.
        future_mask = (aligned.index > t) & (aligned.index <= t_next)
        future = aligned.loc[future_mask]

        if future.empty:
            continue

        future_sector = future[sector_rets.columns]
        future_market = future["market"]

        # Beta de t fica congelado durante todo o mês seguinte.
        future_hedged_daily = future_sector.sub(
            future_market.to_numpy()[:, None] * betas_t.to_numpy()[None, :],
            axis=0,
        )

        future_hedged = future_hedged_daily.sum(axis=0)
        future_raw = future_sector.sum(axis=0)

        row = {
            "date": t,
            "next_date": t_next,

            "resid_rms_corr":
                float(residual_structure_monthly.loc[t, "rms_corr"]),
            "opportunity_score":
                -float(residual_structure_monthly.loc[t, "rms_corr"]),
            "resid_er":
                float(residual_structure_monthly.loc[t, "effective_rank"]),

            "raw_rms_corr":
                float(raw_structure_monthly.loc[t, "rms_corr"]),
            "raw_er":
                float(raw_structure_monthly.loc[t, "effective_rank"]),

            "ic_resid_mom":
                spearman_cross_section(resid_mom, future_hedged),
            "ic_raw_mom":
                spearman_cross_section(raw_mom, future_hedged),

            "spread_resid_mom":
                top_bottom_spread(resid_mom, future_hedged, top_k),
            "spread_raw_mom":
                top_bottom_spread(raw_mom, future_hedged, top_k),

            # Controles descritivos
            "future_market_log_return":
                float(future_market.sum()),
            "future_cross_sectional_dispersion":
                float(future_raw.std()),
        }

        rows.append(row)

    panel = pd.DataFrame(rows).set_index("date").sort_index()
    return panel


alpha_panel = build_monthly_alpha_panel(
    sector_returns,
    market_returns,
    resid_m,
    raw_m,
    lookback=MOMENTUM_LOOKBACK,
    skip=MOMENTUM_SKIP,
    top_k=TOP_K,
)

print("Número total de decisões mensais construídas:", len(alpha_panel))
display(alpha_panel.head())
display(alpha_panel.tail())

# 13. Holdout temporal — ainda fechado

Para reduzir nossa liberdade de escolha, a v0.4 usa uma regra mecânica:

- **70% iniciais das decisões mensais:** Development / Research
- **30% finais:** Holdout OOS

O corte é baseado **somente na ordem temporal**, não em performance.

> A v0.4 **não mostrará resultados do holdout**.
> Ele será aberto apenas em uma versão posterior, depois de decidirmos se a especificação de development está congelada.

Isso significa que podemos abandonar a tese usando apenas development, mas não podemos “consertá-la” depois de abrir o OOS.

In [ ]:
split_idx = int(np.floor(len(alpha_panel) * 0.70))

dev = alpha_panel.iloc[:split_idx].copy()
oos_locked = alpha_panel.iloc[split_idx:].copy()

print("Development:")
print(dev.index.min(), "→", dev.index.max(), "| n =", len(dev))

print()
print("OOS LOCKED:")
print(oos_locked.index.min(), "→", oos_locked.index.max(), "| n =", len(oos_locked))

# IMPORTANTE:
# Não exibimos nenhuma métrica de retorno/IC do OOS nesta versão.

## 14. Primeiro sanity check econômico — Development apenas

Antes de condicionar por Opportunity Set, verificamos se o alpha-base possui **alguma informação cross-sectional no development**.

Isso não exige Sharpe alto.

Queremos apenas saber se o instrumento experimental não é completamente vazio.

Medimos:

\[
E[IC]
\]

e o spread médio top–bottom.

Se o Residual Momentum tiver IC praticamente zero e extremamente instável, ainda podemos testar condicionamento, mas a interpretação de “confiabilidade do alpha” fica mais fraca.

In [ ]:
dev_baseline_summary = pd.Series({
    "n_months": len(dev),

    "mean_IC_resid_mom": dev["ic_resid_mom"].mean(),
    "median_IC_resid_mom": dev["ic_resid_mom"].median(),
    "IC_hit_rate_resid_mom": (dev["ic_resid_mom"] > 0).mean(),

    "mean_IC_raw_mom": dev["ic_raw_mom"].mean(),
    "median_IC_raw_mom": dev["ic_raw_mom"].median(),
    "IC_hit_rate_raw_mom": (dev["ic_raw_mom"] > 0).mean(),

    "mean_spread_resid_mom": dev["spread_resid_mom"].mean(),
    "median_spread_resid_mom": dev["spread_resid_mom"].median(),

    "mean_spread_raw_mom": dev["spread_raw_mom"].mean(),
    "median_spread_raw_mom": dev["spread_raw_mom"].median(),
})

display(dev_baseline_summary)

# 15. Teste principal — Opportunity Score vs. qualidade futura do Residual Momentum

A especificação principal usa:

\[
O_t = -RMSCorr^\epsilon_t
\]

onde valores maiores significam menor dependência residual.

Testaremos em development:

1. relação contínua entre \(O_t\) e \(IC_{t+1}\);
2. relação entre \(O_t\) e spread futuro;
3. comportamento por quintis de \(O_t\).

A hipótese direcional pré-especificada é:

\[
O_t \uparrow
\Rightarrow
IC_{ResidualMomentum,t+1} \uparrow.
\]

In [ ]:
continuous_dev = pd.Series({
    "pearson_opportunity_vs_IC_resid":
        dev["opportunity_score"].corr(dev["ic_resid_mom"], method="pearson"),

    "spearman_opportunity_vs_IC_resid":
        dev["opportunity_score"].corr(dev["ic_resid_mom"], method="spearman"),

    "pearson_opportunity_vs_spread_resid":
        dev["opportunity_score"].corr(dev["spread_resid_mom"], method="pearson"),

    "spearman_opportunity_vs_spread_resid":
        dev["opportunity_score"].corr(dev["spread_resid_mom"], method="spearman"),
})

display(continuous_dev)

In [ ]:
dev_q = dev.copy()

dev_q["opportunity_quintile"] = pd.qcut(
    dev_q["opportunity_score"],
    q=5,
    labels=["Q1", "Q2", "Q3", "Q4", "Q5"],
    duplicates="drop",
)

by_q = (
    dev_q.groupby("opportunity_quintile", observed=True)
         .agg(
             n=("ic_resid_mom", "size"),

             opportunity_mean=("opportunity_score", "mean"),
             residual_dependence_mean=("resid_rms_corr", "mean"),

             IC_resid_mean=("ic_resid_mom", "mean"),
             IC_resid_median=("ic_resid_mom", "median"),
             IC_resid_hit_rate=("ic_resid_mom", lambda s: (s > 0).mean()),

             spread_resid_mean=("spread_resid_mom", "mean"),
             spread_resid_median=("spread_resid_mom", "median"),

             IC_raw_mean=("ic_raw_mom", "mean"),
             spread_raw_mean=("spread_raw_mom", "mean"),
         )
)

display(by_q)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(
    by_q.index.astype(str),
    by_q["IC_resid_mean"],
)
ax.axhline(0.0, linewidth=1)
ax.set_xlabel("Quintil do Opportunity Score")
ax.set_ylabel("Rank IC médio no mês seguinte")
ax.set_title("Development: Opportunity Set vs. Residual Momentum Rank IC")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(
    by_q.index.astype(str),
    by_q["spread_resid_mean"],
)
ax.axhline(0.0, linewidth=1)
ax.set_xlabel("Quintil do Opportunity Score")
ax.set_ylabel("Spread médio futuro: Top 3 - Bottom 3")
ax.set_title("Development: Opportunity Set vs. spread do Residual Momentum")
plt.show()

# 16. Controles — estamos vendo Opportunity Set ou outra coisa trivial?

Ainda no **development**, comparamos a relação usando:

### A. Dependência residual — variável principal
\[
-RMSCorr^\epsilon_t
\]

### B. Dependência bruta
\[
-RMSCorr^{raw}_t
\]

### C. Effective Rank residual
Como ER residual é transformação monotônica de RMS residual, esperamos resultados de ranking praticamente equivalentes.

Isso é proposital: se ER parecer “melhor” apenas por escala, não conta como evidência incremental.

In [ ]:
control_comparison = pd.DataFrame({
    "metric": [
        "Residual Opportunity (-Residual RMS Corr)",
        "Raw Opportunity (-Raw RMS Corr)",
        "Residual Effective Rank",
        "Raw Effective Rank",
    ],
    "spearman_vs_IC_resid_mom": [
        dev["opportunity_score"].corr(dev["ic_resid_mom"], method="spearman"),
        (-dev["raw_rms_corr"]).corr(dev["ic_resid_mom"], method="spearman"),
        dev["resid_er"].corr(dev["ic_resid_mom"], method="spearman"),
        dev["raw_er"].corr(dev["ic_resid_mom"], method="spearman"),
    ],
    "spearman_vs_spread_resid_mom": [
        dev["opportunity_score"].corr(dev["spread_resid_mom"], method="spearman"),
        (-dev["raw_rms_corr"]).corr(dev["spread_resid_mom"], method="spearman"),
        dev["resid_er"].corr(dev["spread_resid_mom"], method="spearman"),
        dev["raw_er"].corr(dev["spread_resid_mom"], method="spearman"),
    ],
})

display(control_comparison)

# 17. Critério de decisão da v0.4

A v0.4 **não procura “significância perfeita”** e não abre o OOS.

## GO provisório
Avançamos para congelar a especificação e abrir OOS somente se o development mostrar conjuntamente:

1. relação economicamente coerente entre Opportunity Score e qualidade do Residual Momentum;
2. efeito não dependente apenas de uma observação extrema;
3. quintis com alguma estrutura/ordenação interpretável;
4. resultado do residual Opportunity Set pelo menos conceitualmente mais interessante que a dependência bruta.

## CONDITIONAL GO
Se a relação existir, mas for fraca ou não monotônica, analisamos estabilidade temporal **sem alterar 12–1 ou escolher novos parâmetros**.

## NO-GO
Encerramos a tese se:

- IC/spread forem essencialmente aleatórios em todos os estados;
- relação mudar de sinal sem padrão;
- somente a métrica bruta explicar o resultado;
- o efeito depender de poucos episódios extremos.

---

## Importante

Não abra ou resuma `oos_locked` antes da decisão.

A próxima versão só deve existir depois de avaliarmos juntos os outputs de development.

## Registro — v0.4

| Item | Decisão congelada |
|---|---|
| Tese | Residual Dependence / Opportunity Set |
| Medida principal | `- Residual RMS Correlation` |
| ER | transformação interpretável, não fonte independente |
| Alpha principal | Residual Momentum |
| Lookback | 12–1 |
| Controle de alpha | Momentum bruto 12–1 |
| Métrica primária | Cross-sectional Rank IC |
| Métrica econômica secundária | Top 3 – Bottom 3 spread |
| Frequência | mensal |
| Beta hedge futuro | beta estimado em \(t\) e congelado |
| Development | 70% inicial da amostra |
| OOS | 30% final, **LOCKED** |
| Threshold de trading | nenhum |
| Carteira final | ainda não construída |

# 18. v0.5 — Adaptive Factor Neutralization

## Correção metodológica importante

Na v0.4 nós já observamos **estatísticas agregadas de todo o período 2001–2018**.

Portanto, seria incorreto dividir agora esse mesmo intervalo em “design” e “internal validation” e chamar a parte final de validação limpa.

Ela **já foi parcialmente contaminada pelo processo de pesquisa**.

Assim, a estrutura temporal correta passa a ser:

\[
\boxed{
2001\text{–}2018 = Research / Exploratory Sample
}
\]

\[
\boxed{
2018\text{–}2026 = Final OOS realmente intocado
}
\]

Podemos dividir 2001–2018 em subperíodos para testar **consistência interna**, mas isso não será tratado como confirmação independente.

> A confirmação real da nova hipótese só poderá vir do OOS final, que continua fechado nesta versão.

## 19. Nova hipótese congelada antes de abrir o OOS

O resultado exploratório mais promissor da v0.4 foi:

\[
ResidualMomentum_{12-1}
\]

ter apresentado maior capacidade cross-sectional que o momentum bruto no research sample.

Isso motivou uma nova pergunta:

> **Quando neutralizar o fator de mercado melhora um sinal de momentum?**

Para cada setor:

\[
r_{i,t}
=
\alpha_{i,t}
+
\beta_{i,t}r_{SPY,t}
+
\epsilon_{i,t}.
\]

Definimos a **Market Commonality** em \(t\) como:

\[
Commonality_t
=
\frac{1}{N}
\sum_{i=1}^{N}R^2_{i,t},
\]

onde cada \(R^2_{i,t}\) vem do market model estimado com os 252 pregões disponíveis até \(t\).

A variável-alvo principal é:

\[
\Delta IC_{t+1}
=
IC^{ResidualMom}_{t+1}
-
IC^{RawMom}_{t+1}.
\]

### Hipótese direcional pré-especificada

\[
\boxed{
Commonality_t \uparrow
\Rightarrow
\Delta IC_{t+1} \uparrow
}
\]

Interpretação:

> Quanto mais o fator de mercado domina os movimentos dos setores, maior deveria ser o valor incremental de retirar esse fator antes de construir o ranking de momentum.

### Métrica econômica secundária

\[
\Delta Spread_{t+1}
=
Spread^{ResidualMom}_{t+1}
-
Spread^{RawMom}_{t+1}.
\]

Nenhum threshold de trading será criado nesta versão.

## 20. Por que \(R^2\) é preferido aqui

Nesta hipótese, não estamos tentando representar a “geometria” do mercado.

Queremos responder diretamente:

> **Quanto do movimento dos setores está sendo explicado pelo fator de mercado?**

O \(R^2\) do market model é uma medida simples e auditável para isso.

Isso também evita recuperar o Effective Rank apenas para preservar uma narrativa anterior.

In [ ]:
def market_model_r2(
    sector_window: pd.DataFrame,
    market_window: pd.Series,
) -> pd.Series:
    """
    R² do market model por ativo, estimado somente com a janela histórica recebida.
    """
    _, _, residuals = fit_market_model(sector_window, market_window)

    r2 = {}

    for ticker in sector_window.columns:
        y = sector_window[ticker]
        resid = residuals[ticker]

        sse = float(np.square(resid).sum())
        sst = float(np.square(y - y.mean()).sum())

        r2[ticker] = np.nan if sst <= 0 else 1.0 - sse / sst

    return pd.Series(r2, dtype=float)


def build_market_commonality_series(
    sector_rets: pd.DataFrame,
    market_rets: pd.Series,
    decision_dates: pd.DatetimeIndex,
    lookback: int = 252,
) -> pd.DataFrame:
    aligned = sector_rets.join(
        market_rets.rename("market"),
        how="inner"
    ).dropna()

    rows = []

    for t in decision_dates:
        if t not in aligned.index:
            continue

        pos = aligned.index.get_loc(t)

        if not isinstance(pos, (int, np.integer)) or pos + 1 < lookback:
            continue

        hist = aligned.iloc[pos - lookback + 1 : pos + 1]
        sector_hist = hist[sector_rets.columns]
        market_hist = hist["market"]

        r2_by_asset = market_model_r2(sector_hist, market_hist)

        rows.append({
            "date": t,
            "market_commonality_mean_r2": float(r2_by_asset.mean()),
            "market_commonality_median_r2": float(r2_by_asset.median()),
            "market_commonality_min_r2": float(r2_by_asset.min()),
            "market_commonality_max_r2": float(r2_by_asset.max()),
            "market_commonality_std_r2": float(r2_by_asset.std()),
        })

    return pd.DataFrame(rows).set_index("date").sort_index()


commonality = build_market_commonality_series(
    sector_returns,
    market_returns,
    alpha_panel.index,
    lookback=MOMENTUM_LOOKBACK,
)

display(commonality.head())
display(commonality.tail())

## 21. Painel da nova hipótese

O painel abaixo usa **apenas variáveis já congeladas**:

- Residual Momentum 12–1;
- Raw Momentum 12–1;
- Rank IC;
- spread Top 3 – Bottom 3;
- Market Commonality via mean \(R^2\).

Não alteramos o alpha depois de observar o resultado da v0.4.

In [ ]:
adaptive_panel = alpha_panel.join(commonality, how="inner").copy()

adaptive_panel["delta_ic"] = (
    adaptive_panel["ic_resid_mom"]
    - adaptive_panel["ic_raw_mom"]
)

adaptive_panel["delta_spread"] = (
    adaptive_panel["spread_resid_mom"]
    - adaptive_panel["spread_raw_mom"]
)

# Mantém a mesma separação temporal já criada na v0.4.
research = adaptive_panel.loc[dev.index].copy()
final_oos_locked = adaptive_panel.loc[oos_locked.index].copy()

print("RESEARCH SAMPLE:")
print(research.index.min(), "→", research.index.max(), "| n =", len(research))

print()
print("FINAL OOS LOCKED:")
print(final_oos_locked.index.min(), "→", final_oos_locked.index.max(), "| n =", len(final_oos_locked))

# Nenhuma métrica do OOS é exibida nesta versão.

# 22. Diagnóstico exploratório no Research Sample

Tudo abaixo é **exploratório**, porque a hipótese nasceu após observar resultados desse mesmo intervalo.

O objetivo é decidir se a relação é coerente o suficiente para justificar **uma única abertura posterior do OOS**.

Não usaremos esta etapa para escolher parâmetros.

In [ ]:
research_summary = pd.Series({
    "n_months": len(research),

    "commonality_mean":
        research["market_commonality_mean_r2"].mean(),

    "commonality_std":
        research["market_commonality_mean_r2"].std(),

    "delta_IC_mean":
        research["delta_ic"].mean(),

    "delta_IC_median":
        research["delta_ic"].median(),

    "delta_IC_positive_rate":
        (research["delta_ic"] > 0).mean(),

    "delta_spread_mean":
        research["delta_spread"].mean(),

    "delta_spread_median":
        research["delta_spread"].median(),
})

display(research_summary)

## 23. Teste contínuo exploratório

Hipótese:

\[
Commonality_t \uparrow
\Rightarrow
\Delta IC_{t+1} \uparrow.
\]

O Spearman é a métrica primária porque não exige relação linear.

In [ ]:
adaptive_continuous = pd.Series({
    "pearson_commonality_vs_delta_IC":
        research["market_commonality_mean_r2"].corr(
            research["delta_ic"],
            method="pearson",
        ),

    "spearman_commonality_vs_delta_IC":
        research["market_commonality_mean_r2"].corr(
            research["delta_ic"],
            method="spearman",
        ),

    "pearson_commonality_vs_delta_spread":
        research["market_commonality_mean_r2"].corr(
            research["delta_spread"],
            method="pearson",
        ),

    "spearman_commonality_vs_delta_spread":
        research["market_commonality_mean_r2"].corr(
            research["delta_spread"],
            method="spearman",
        ),
})

display(adaptive_continuous)

## 24. Quintis de Market Commonality — exploratório

Se a hipótese tiver alguma estrutura, esperamos que o benefício relativo da neutralização tenda a crescer à medida que a commonality aumenta.

Não exigimos monotonicidade perfeita, mas uma forma completamente errática enfraquece muito a tese.

In [ ]:
research_q = research.copy()

research_q["commonality_quintile"] = pd.qcut(
    research_q["market_commonality_mean_r2"],
    q=5,
    labels=["Q1", "Q2", "Q3", "Q4", "Q5"],
    duplicates="drop",
)

commonality_by_q = (
    research_q.groupby("commonality_quintile", observed=True)
              .agg(
                  n=("delta_ic", "size"),

                  commonality_mean=(
                      "market_commonality_mean_r2",
                      "mean"
                  ),

                  delta_IC_mean=("delta_ic", "mean"),
                  delta_IC_median=("delta_ic", "median"),
                  delta_IC_positive_rate=(
                      "delta_ic",
                      lambda s: (s > 0).mean()
                  ),

                  delta_spread_mean=("delta_spread", "mean"),
                  delta_spread_median=("delta_spread", "median"),

                  IC_resid_mean=("ic_resid_mom", "mean"),
                  IC_raw_mean=("ic_raw_mom", "mean"),
              )
)

display(commonality_by_q)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(
    commonality_by_q.index.astype(str),
    commonality_by_q["delta_IC_mean"],
)
ax.axhline(0.0, linewidth=1)
ax.set_xlabel("Quintil de Market Commonality")
ax.set_ylabel("Δ Rank IC = Residual Momentum - Raw Momentum")
ax.set_title("Research sample: benefício da neutralização por commonality")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(
    research["market_commonality_mean_r2"],
    research["delta_ic"],
    alpha=0.65,
)
ax.axhline(0.0, linewidth=1)
ax.set_xlabel("Market Commonality (mean R²)")
ax.set_ylabel("Δ Rank IC")
ax.set_title("Market Commonality vs. ganho de Rank IC")
plt.show()

# 25. Consistência temporal dentro do Research Sample

Como todo o intervalo 2001–2018 já foi parcialmente observado durante o research, isto **não é validação independente**.

Dividimos mecanicamente o research sample em três blocos cronológicos apenas para verificar se a relação está concentrada em uma única época.

In [ ]:
# Divisão cronológica robusta usando iloc.
# Evitamos np.array_split(DataFrame, 3), que em algumas combinações
# pandas/numpy pode retornar objetos sem índice pandas.

n = len(research)
cut1 = n // 3
cut2 = 2 * n // 3

research_blocks = [
    research.iloc[:cut1].copy(),
    research.iloc[cut1:cut2].copy(),
    research.iloc[cut2:].copy(),
]

block_rows = []

for idx, block in enumerate(research_blocks, start=1):
    block_rows.append({
        "block": f"B{idx}",
        "start": block.index.min(),
        "end": block.index.max(),
        "n": len(block),

        "mean_commonality":
            block["market_commonality_mean_r2"].mean(),

        "mean_delta_IC":
            block["delta_ic"].mean(),

        "median_delta_IC":
            block["delta_ic"].median(),

        "spearman_commonality_vs_delta_IC":
            block["market_commonality_mean_r2"].corr(
                block["delta_ic"],
                method="spearman",
            ),

        "mean_delta_spread":
            block["delta_spread"].mean(),
    })

temporal_consistency = pd.DataFrame(block_rows).set_index("block")
display(temporal_consistency)

# 26. Controle: Market Commonality é só correlação bruta renomeada?

Não precisamos que Commonality seja completamente independente da correlação bruta.

Mas devemos saber o quanto elas se sobrepõem.

Isso ajuda a decidir se o modelo de fatores oferece interpretação adicional ou apenas uma nova escala para o mesmo fenômeno.

In [ ]:
commonality_controls = pd.Series({
    "pearson_commonality_vs_raw_RMS_corr":
        research["market_commonality_mean_r2"].corr(
            research["raw_rms_corr"],
            method="pearson",
        ),

    "spearman_commonality_vs_raw_RMS_corr":
        research["market_commonality_mean_r2"].corr(
            research["raw_rms_corr"],
            method="spearman",
        ),

    "pearson_commonality_vs_residual_RMS_corr":
        research["market_commonality_mean_r2"].corr(
            research["resid_rms_corr"],
            method="pearson",
        ),

    "spearman_commonality_vs_residual_RMS_corr":
        research["market_commonality_mean_r2"].corr(
            research["resid_rms_corr"],
            method="spearman",
        ),
})

display(commonality_controls)

# 27. Critério para decidir se vale gastar o Final OOS

Como o research sample é exploratório, **não existe confirmação aqui**.

A decisão é apenas:

> A hipótese está suficientemente coerente para justificar uma abertura única do OOS?

## Candidato a GO para abertura do OOS

A relação deve, no mínimo:

1. ter sinal positivo no Spearman `Commonality → ΔIC`;
2. apresentar interpretação coerente nos quintis;
3. não estar concentrada inteiramente em um único bloco temporal;
4. mostrar algum benefício econômico secundário em `ΔSpread`;
5. não exigir mudança do 12–1, do SPY, dos 252 dias ou da métrica após observar resultados.

## NO-GO sem abrir OOS

Se a relação for praticamente zero, de sinal contrário ou estruturalmente errática, abandonamos Adaptive Factor Neutralization **sem gastar o holdout final**.

## Regra crítica

Se decidirmos abrir o OOS na próxima versão:

- será aberto **uma única vez**;
- não mudaremos a especificação depois;
- resultado negativo encerrará a tese;
- não voltaremos para 6–1, outro benchmark ou outro threshold.

## Registro de governança — v0.5

| Item | Status |
|---|---|
| Nova tese | Adaptive Factor Neutralization |
| Alpha residual | Residual Momentum 12–1 |
| Controle | Raw Momentum 12–1 |
| Fator | SPY |
| Janela do market model | 252 pregões |
| Variável explicativa | Mean \(R^2\) cross-sectional |
| Target primário | \(\Delta IC\) |
| Target secundário | \(\Delta Spread\) |
| 2001–2018 | Research sample já contaminado/exploratório |
| 2018–2026 | **Final OOS intocado** |
| Threshold de trading | nenhum |
| Carteira | ainda não construída |
| Próxima decisão | abrir ou não o Final OOS |

> A v0.5 não produz nenhuma métrica do OOS.